### Make a temporary space in /flash

In [1]:
import os, random, string

In [2]:
cwd = os.path.basename(os.getcwd())
tmpdir = os.path.join("/flash/ReiterU/tmp/", 'tomo_' + ''.join(random.choices(string.ascii_uppercase, k=12)))
print(tmpdir)
if not os.path.exists(tmpdir):
    os.makedirs(tmpdir)

/flash/ReiterU/tmp/tomo_FIDDIAGUICDU


### PARAMETERS

In [3]:
#### path to relevant files
rootdir = "/bucket/ReiterU/OLdev/experiments/20250522_squid_cal520"
recname = "rec4"
savedir = "/bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250522_rec4"

templatefile = os.path.join(rootdir, "2P", recname, "run2/mcorr_nonrigid/QC/QC_nonrigid/template.tif")
roifile = os.path.join(rootdir, "2P", recname, "ROI_transformed_corrected_renamed.zip")
dfffile = os.path.join(rootdir, "2P", recname, "dff_ROI_transformed_corrected_renamed/DFF.hdf5")
syncfile = os.path.join(rootdir, "sync", f"{recname}.tdms")

QI_THRESH = 0.25

### Import stuff

In [4]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from scipy import signal
import h5py
import subprocess

In [5]:
import sys
sys.path.append("../../../")
from calfuncs.io import read_sync_ttl
from calfuncs.full_field_step import get_ett2, plot_chirp_eta, compute_qi

In [6]:
from matplotlib import font_manager
import matplotlib as mpl

font_manager.fontManager.addfont('/bucket/ReiterU/common/fonts/arial.ttf')
mpl.rcParams.update({
    'font.size': 7,
    'font.family': 'sans-serif',
    'font.sans-serif': "Arial",
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
    'axes.linewidth': 0.5,
    'xtick.major.width': 0.5,
    'xtick.major.size': 2,
    'xtick.minor.size': 2,
    'xtick.minor.width': 0.5,
    'ytick.major.width': 0.5,
    'ytick.minor.width': 0.5,
    'ytick.major.size': 2,
    'ytick.minor.size': 2,
})

### Load TDMS file

In [7]:
sync_ttl = read_sync_ttl(syncfile)
twop_ttl = sync_ttl["twop_ttl"]
twop_re = sync_ttl["twop_re"]
vis_re = sync_ttl["vis_re"]
print(twop_ttl.shape, twop_re.shape, vis_re.shape)

(722500,) (10997,) (60,)


### Load DF/F

In [8]:
with h5py.File(dfffile, "r") as hf:
    F = hf["F"][:]
    dff = hf["dff"][:]
print(F.shape) # F is sampled at 1 kH

(241, 722500)


### Chirp

In [9]:
E = []
for nid in np.arange(0, F.shape[0]):
    ett1 = get_ett2(F, nid, vis_re[0::6], window=[-2000, 19000], baseline_window=[-4000, 70000], subsampling=50)
    ett1["session"] = 0
    ett2 = get_ett2(F, nid, vis_re[2::6], window=[-2000, 19000], baseline_window=[-27000, 43000], subsampling=50)
    ett2["session"] = 1
    ett3 = get_ett2(F, nid, vis_re[4::6], window=[-2000, 19000], baseline_window=[-51000, 19000], subsampling=50)
    ett3["session"] = 2
    ett = pd.concat([ett1, ett2, ett3])
    ett["neuron"] = nid
    ett = ett.set_index(["neuron", "session", "trial", "time"])
    E.append(ett)
E = pd.concat(E)

In [ ]:
dt = 1e-2
stimulus = np.zeros(int(21.0/dt), dtype=np.float32)
stimulus[0:int(2.0/dt)] = 0
stimulus[int(2.0/dt):int(6.0/dt)] = 1
stimulus[int(6.0/dt):int(10.0/dt)] = 0
stimulus[int(10.0/dt):int(12.0/dt)] = 0.5
stimulus[int(12.0/dt):int(20.0/dt)] = 0.5 + 0.5 * signal.chirp(np.linspace(0, 8, int(20.0/dt)-int(12.0/dt)), f0=0.5, f1=10, t1=8, method='logarithmic', phi=90)
stimulus[int(20.0/dt):int(21.0/dt)] = 0

for nid in np.arange(0, dff.shape[0]):
    qi = compute_qi(E, nid)
    print("Neuron:", nid, "QI:", qi)
    savename = os.path.join(tmpdir, f"ETA_{nid:03d}.png")
    plot_chirp_eta(E, nid, stimulus, vmin=-2, vmax=3,  cmap_name="bone_r", figsize=(5,2.0), show_plot=True, figsavename=savename)

### Compute trial-average

In [11]:
R = []
neurons = E.index.get_level_values("neuron").unique()

for n in neurons:
    M = []
    df2 = E.xs(n, level="neuron")
    for i in range(3):
        df3 = df2.xs(i, level="session")
        mat = df3.reset_index().pivot(index="trial", columns="time", values="y").to_numpy()
        M.append(mat)
    M = np.hstack(M)
    M = M[~np.all(np.isnan(M), axis=1)] # remove trials with NaN values
    qi = compute_qi(E, n)
    y_mean = np.nanmean(M, axis=0)
    row = {
        "recording": 0,
        "neuron": n,
        "qi": qi,
    }
    for j, value in enumerate(y_mean):
        row[f"time_{j}"] = value

    R.append(row)

R = pd.DataFrame(R)

In [12]:
display(R)

,recording,neuron,qi,time_0,time_1,time_2,time_3,time_4,time_5,time_6,...,time_1250,time_1251,time_1252,time_1253,time_1254,time_1255,time_1256,time_1257,time_1258,time_1259
0,0,0,0.515426,1.201130,1.425307,1.267680,1.203863,1.245130,1.043607,0.898859,...,-0.867244,-0.815808,-0.936542,-1.233145,-1.016522,-0.805797,-0.902529,-1.104139,-0.880924,-1.185932
1,0,1,0.584246,-0.112104,0.271543,-0.233879,-0.508096,-0.644812,-0.310061,-0.271602,...,0.906694,0.600824,0.396635,0.393564,0.537604,0.237044,0.315288,-0.026510,-0.052803,0.095784
2,0,2,0.672618,0.376715,0.187273,0.296640,0.247845,0.213843,0.432610,0.435460,...,-1.335398,-1.426259,-1.088441,-1.035211,-1.148719,-1.075307,-1.043108,-1.137600,-0.956029,-0.930589
3,0,3,0.670741,-0.289624,-0.363240,-0.105109,-0.185100,-0.223350,-0.215137,-0.309463,...,-1.130345,-1.108720,-1.204614,-1.090072,-0.988250,-0.990286,-1.196655,-1.338517,-1.310913,-1.148709
4,0,4,0.773148,0.196667,0.065708,0.082298,0.181784,0.189814,0.138042,0.022854,...,-0.643316,-0.714066,-0.623769,-0.630953,-0.718061,-0.733798,-0.782292,-0.827539,-0.781104,-0.737852
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
236,0,236,0.269322,1.006299,0.958801,0.786085,0.181002,0.113360,0.252613,0.583497,...,-0.702276,-0.587524,-0.666946,-0.752919,-0.574241,-0.528926,-0.753812,-0.681307,-0.579579,-0.689080
237,0,237,0.757358,-0.141556,-0.225100,0.023965,0.106336,0.164958,0.236038,0.159669,...,-1.016030,-1.095629,-1.050372,-1.042353,-1.013596,-1.166793,-1.073033,-1.053838,-1.079601,-1.220077
238,0,238,0.305991,0.301647,0.476728,0.095837,0.063960,0.080241,0.360466,0.496032,...,-0.820023,-0.919148,-0.767709,-0.794651,-0.553361,-0.280232,-0.471221,-0.264957,-0.472189,-0.635871
239,0,239,0.167731,-0.209694,0.129229,-0.304540,-0.054881,-0.186270,-0.041427,-0.082126,...,-0.925361,-0.506133,-0.437675,-0.449726,-0.322299,-0.511355,-0.006658,-0.111122,-0.425038,-0.623007


In [13]:
with pd.HDFStore(os.path.join(tmpdir, "response_table.hdf5"), mode='w') as store:
    store.put('E', E)
    store.put('responses', R)

### Copy data to bucket

In [14]:
cmd1 = f'ssh deigo "mkdir -p {savedir}"'
cmd2 = f'ssh deigo "cp -r {tmpdir}/* {savedir}"'
print(cmd1)
print(cmd2)

subprocess.run(cmd1, shell=True, check=True)
subprocess.run(cmd2, shell=True, check=True)

ssh deigo "mkdir -p /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250522_rec4"
ssh deigo "cp -r /flash/ReiterU/tmp/tomo_FIDDIAGUICDU/* /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250522_rec4"


Offending key for IP in /home/t/tomoyuki-mano1/.ssh/known_hosts:2
Matching host key in /home/t/tomoyuki-mano1/.ssh/known_hosts:6
Offending key for IP in /home/t/tomoyuki-mano1/.ssh/known_hosts:2
Matching host key in /home/t/tomoyuki-mano1/.ssh/known_hosts:6


CompletedProcess(args='ssh deigo "cp -r /flash/ReiterU/tmp/tomo_FIDDIAGUICDU/* /bucket/ReiterU/OLdev/analysis/20250528_IGL_monitor_flip_analysis/20250522_rec4"', returncode=0)